In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  },
  "colab": {
   "provenance": [],
   "gpuType": "T4"
  },
  "accelerator": "GPU"
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 📄 Step 1 — Data Collection\n",
    "\n",
    "**Run this notebook on Google Colab.**\n",
    "\n",
    "This notebook:\n",
    "1. Mounts Google Drive\n",
    "2. Clones your GitHub repo\n",
    "3. Runs all data collection scripts\n",
    "4. Saves output to Google Drive\n",
    "\n",
    "---\n",
    "> ⚠️ Before running: Replace `YOUR_USERNAME` with your GitHub username."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 1 — Mount Google Drive"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "print('Google Drive mounted successfully!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 2 — Install Dependencies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install requests beautifulsoup4 lxml pypdf \\\n",
    "             indic-nlp-library sentencepiece \\\n",
    "             loguru rank-bm25 langdetect \\\n",
    "             chromadb sentence-transformers -q\n",
    "\n",
    "print('Dependencies installed!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 3 — Clone Repository"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "\n",
    "# Replace with your GitHub username\n",
    "GITHUB_USERNAME = 'YOUR_USERNAME'\n",
    "REPO_NAME       = 'kannada-legal-ai'\n",
    "\n",
    "# Clone the repo\n",
    "if not os.path.exists(f'/content/{REPO_NAME}'):\n",
    "    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git\n",
    "else:\n",
    "    print('Repo already exists. Pulling latest...')\n",
    "    !cd /content/{REPO_NAME} && git pull\n",
    "\n",
    "%cd /content/{REPO_NAME}\n",
    "print(f'Working directory: {os.getcwd()}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 4 — Scrape IPC Sections"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '/content/kannada-legal-ai')\n",
    "\n",
    "!python -m data.collection.scrape_india_code\n",
    "\n",
    "# Verify output\n",
    "import json\n",
    "with open('data/raw/ipc_sections/ipc_sections.json') as f:\n",
    "    ipc = json.load(f)\n",
    "print(f'IPC sections collected: {len(ipc)}')\n",
    "print(f'Sample: {ipc[0][\"section_number\"]} — {ipc[0][\"title\"]}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 5 — Scrape Karnataka Laws"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python -m data.collection.scrape_karnataka_gov\n",
    "\n",
    "# Verify output\n",
    "with open('data/raw/karnataka_state_laws/karnataka_laws.json') as f:\n",
    "    ka = json.load(f)\n",
    "print(f'Karnataka laws collected: {len(ka)}')\n",
    "\n",
    "with open('data/raw/legal_aid_pamphlets/vikaspedia_kn.json') as f:\n",
    "    vika = json.load(f)\n",
    "print(f'Vikaspedia articles collected: {len(vika)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 6 — Translate to Kannada (GPU accelerated)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Install IndicTrans2 for better translation quality on GPU\n",
    "!pip install IndicTransToolkit -q\n",
    "\n",
    "# Run translation with Colab GPU mode\n",
    "!python -m data.collection.translate_to_kannada --mode colab\n",
    "\n",
    "# Verify\n",
    "import os\n",
    "for f in os.listdir('data/processed/'):\n",
    "    path = f'data/processed/{f}'\n",
    "    size = os.path.getsize(path)\n",
    "    print(f'  {f} — {size} bytes')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 7 — Generate QA Pairs"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python -m data.collection.generate_qa_pairs\n",
    "\n",
    "# Verify splits\n",
    "for split in ['train', 'val', 'test']:\n",
    "    path = f'data/annotated/{split}/qa_pairs.jsonl'\n",
    "    with open(path) as f:\n",
    "        count = sum(1 for _ in f)\n",
    "    print(f'{split:10} : {count} pairs')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 8 — Build Vector Store"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python -m scripts.build_vector_store\n",
    "\n",
    "# Verify vector store\n",
    "from rag.vector_store import get_collection_stats\n",
    "stats = get_collection_stats()\n",
    "print(f\"Vector store documents: {stats['total_documents']}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 9 — Save All Data to Google Drive"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "import os\n",
    "\n",
    "DRIVE_PATH = '/content/drive/MyDrive/kannada-legal-ai'\n",
    "os.makedirs(DRIVE_PATH, exist_ok=True)\n",
    "\n",
    "# Save processed data\n",
    "shutil.copytree(\n",
    "    'data/processed',\n",
    "    f'{DRIVE_PATH}/data/processed',\n",
    "    dirs_exist_ok=True\n",
    ")\n",
    "\n",
    "# Save annotated data\n",
    "shutil.copytree(\n",
    "    'data/annotated',\n",
    "    f'{DRIVE_PATH}/data/annotated',\n",
    "    dirs_exist_ok=True\n",
    ")\n",
    "\n",
    "# Save vector store\n",
    "shutil.copytree(\n",
    "    'data/vector_store',\n",
    "    f'{DRIVE_PATH}/data/vector_store',\n",
    "    dirs_exist_ok=True\n",
    ")\n",
    "\n",
    "print('All data saved to Google Drive!')\n",
    "print(f'Location: {DRIVE_PATH}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 10 — Push Data Back to GitHub"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Configure git\n",
    "!git config user.email 'your@email.com'\n",
    "!git config user.name  'Your Name'\n",
    "\n",
    "# Stage and commit\n",
    "!git add data/annotated/ data/processed/\n",
    "!git commit -m 'add collected and translated legal data'\n",
    "\n",
    "# Push (you may need a personal access token)\n",
    "# If asked for password use your GitHub personal access token\n",
    "!git push origin main\n",
    "\n",
    "print('Data pushed to GitHub!')"
   ]
  }
 ]
}